# Step1: Install packages
## 1.1 install plink2

In [ ]:
# Install plink2
wget -q https://s3.amazonaws.com/plink2-assets/plink2_linux_avx2_20260110.zip
unzip -o plink2_linux_avx2_20260110.zip
./plink2 --version

## 1.2 Load the Clinvar dataset

In [ ]:
wget ftp://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz
wget ftp://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz.tbi

# Step2: Indentify the genotype for Genes
## 2.1 Strict: only use the Pathogenic/Likely_pathogenic mutations

In [ ]:
# #!/bin/bash

# ===== GENE LIST =====
genes=(
ALS2  ANG    ANXA11 AR ARHGEF28 ARPP21 ATXN2 C9orf72 CAV1 CAV2 CCNF CFAP410 CHCHD10 CHMP2B CYLD
DCTN1 DNAJC7 ERBB4 ERLIN1 ERLIN2 FIG4 FUS GLE1 GLT8D1 GRN HNRNPA2B1 KIF5A LGALSL LRP12 MATR3
NEFH NEK1 NUP50 OPTN PFN1 PRPH SLC52A2 SLC52A3 SOD1 SPG11 SPTLC2 SQSTM1 SS18L1 TAF15 TARDBP
TBK1 TIA1 TUBA4A UBQLN2 VAPB VCP HNRNPA1 SPTLC1 UNC13A
)

# ===== CONFIGURATION =====
CLINVAR_FILE="clinvar.vcf.gz"
PLINK_BASE_DIR="/mnt/project/Bulk/DRAGEN WGS/DRAGEN population level WGS variants, PLINK format [500k release]"

# Create directories
WORKING_DIR="gene_extraction_working"
RESULTS_DIR="gene_extraction_results"
mkdir -p "$WORKING_DIR" "$RESULTS_DIR"

SUMMARY_CSV="gene_extraction_summary.csv"

# Create summary CSV with header
echo "Gene_Name,Chromosome,Pathogenic_Variants,Carriers" > "$SUMMARY_CSV"

# ===== MAIN LOOP =====
for GENE_NAME in "${genes[@]}"; do
    echo ""
    echo "================================"
    echo "Processing gene: $GENE_NAME"
    echo "================================"
    
    OUTPUT_PREFIX="UKB_${GENE_NAME,,}_pathogenic"
    
    # ===== STEP 1: Extract gene variants from ClinVar =====
    echo "Step 1: Extracting $GENE_NAME variants from ClinVar..."
    if ! zgrep "GENEINFO=${GENE_NAME}:" "$CLINVAR_FILE" > "$WORKING_DIR/${GENE_NAME}_clinvar_by_name.vcf" 2>/dev/null; then
        echo "  ERROR: Could not extract $GENE_NAME from ClinVar"
        continue
    fi
    
    VARIANT_COUNT=$(grep -v "^#" "$WORKING_DIR/${GENE_NAME}_clinvar_by_name.vcf" | wc -l)
    echo "  Found $VARIANT_COUNT variants"
    
    # ===== STEP 2: Filter for pathogenic/likely pathogenic =====
    echo "Step 2: Filtering for pathogenic variants..."
    grep -E "CLNSIG=Pathogenic|CLNSIG=Likely_pathogenic" "$WORKING_DIR/${GENE_NAME}_clinvar_by_name.vcf" > "$WORKING_DIR/${GENE_NAME}_pathogenic_variants.vcf"
    
    PATHOGENIC_COUNT=$(grep -v "^#" "$WORKING_DIR/${GENE_NAME}_pathogenic_variants.vcf" | wc -l)
    echo "  Found $PATHOGENIC_COUNT pathogenic variants"
    
    # ===== STEP 3: Extract chromosome from ClinVar file =====
    echo "Step 3: Extracting chromosome..."
    GENE_CHROM=$(awk '!/^#/ {print $1; exit}' "$WORKING_DIR/${GENE_NAME}_clinvar_by_name.vcf")
    echo "  Detected chromosome: $GENE_CHROM"
    
    if [[ $PATHOGENIC_COUNT -eq 0 ]]; then
        echo "  WARNING: No pathogenic variants found for $GENE_NAME"
        echo ""
        echo " COMPLETE: $GENE_NAME"
        echo "  Chromosome: $GENE_CHROM"
        echo "  Pathogenic variants: 0"
        echo "  Carriers identified: 0"
        echo "$GENE_NAME,$GENE_CHROM,0,0" >> "$SUMMARY_CSV"
        continue
    fi
    
    # ===== STEP 4: Map chromosome to PLINK file code =====
    echo "Step 4: Mapping chromosome to PLINK file..."
    PLINK_FILEROOT="${PLINK_BASE_DIR}/ukb24308_c${GENE_CHROM}_b0_v1"
    echo "  PLINK file: ukb24308_c${GENE_CHROM}_b0_v1"
    
    # Check if PLINK files exist
    if [[ ! -f "${PLINK_FILEROOT}.pgen" ]]; then
        echo "  ERROR: PLINK files not found for chromosome $GENE_CHROM"
        echo "$GENE_NAME,$GENE_CHROM,$PATHOGENIC_COUNT,ERROR" >> "$SUMMARY_CSV"
        continue
    fi
    
    # ===== STEP 5: Create PLINK extraction IDs =====
    echo "Step 5: Creating PLINK extraction IDs..."
    awk '!/^#/ {print "DRAGEN:chr"$1":"$2":"$4":"$5}' "$WORKING_DIR/${GENE_NAME}_pathogenic_variants.vcf" > "$WORKING_DIR/${GENE_NAME}_pathogenic_ids_for_extraction.txt"
    
    # ===== STEP 6: Extract variants from PLINK files =====
    echo "Step 6: Extracting variants from PLINK files..."
    if ! ./plink2 \
        --pfile "$PLINK_FILEROOT" \
        --extract "$WORKING_DIR/${GENE_NAME}_pathogenic_ids_for_extraction.txt" \
        --no-psam-pheno \
        --make-bed \
        --out "$WORKING_DIR/${OUTPUT_PREFIX}_carriers" > "$WORKING_DIR/${OUTPUT_PREFIX}_carriers.log" 2>&1; then
        echo "  ERROR: PLINK extraction failed for $GENE_NAME"
        echo "$GENE_NAME,$GENE_CHROM,$PATHOGENIC_COUNT,ERROR" >> "$SUMMARY_CSV"
        continue
    fi
    
    # Check if any variants were extracted
    if [[ ! -f "$WORKING_DIR/${OUTPUT_PREFIX}_carriers.bim" ]] || [[ $(wc -l < "$WORKING_DIR/${OUTPUT_PREFIX}_carriers.bim") -eq 0 ]]; then
        echo "  WARNING: Pathogenic variants found ($PATHOGENIC_COUNT) but no individuals carry them"
        echo ""
        echo "✓ COMPLETE: $GENE_NAME"
        echo "  Chromosome: $GENE_CHROM"
        echo "  Pathogenic variants: $PATHOGENIC_COUNT"
        echo "  Carriers identified: 0"
        echo "$GENE_NAME,$GENE_CHROM,$PATHOGENIC_COUNT,0" >> "$SUMMARY_CSV"
        continue
    fi
    
    # ===== STEP 7: Prepare allele recoding file =====
    echo "Step 7: Preparing allele recoding..."
    awk -F: '{print $0"\t"$5}' "$WORKING_DIR/${GENE_NAME}_pathogenic_ids_for_extraction.txt" > "$WORKING_DIR/${GENE_NAME}_alleles_to_count.txt"
    
    # ===== STEP 8: Recode to additive format =====
    echo "Step 8: Recoding to additive format..."
    if ! ./plink2 \
        --bfile "$WORKING_DIR/${OUTPUT_PREFIX}_carriers" \
        --recode A \
        --recode-allele "$WORKING_DIR/${GENE_NAME}_alleles_to_count.txt" \
        --out "$WORKING_DIR/${OUTPUT_PREFIX}_corrected" > "$WORKING_DIR/${OUTPUT_PREFIX}_corrected.log" 2>&1; then
        echo "  ERROR: PLINK recoding failed for $GENE_NAME"
        echo "$GENE_NAME,$GENE_CHROM,$PATHOGENIC_COUNT,ERROR" >> "$SUMMARY_CSV"
        continue
    fi
    
    # ===== STEP 9: Copy corrected RAW file to results =====
    echo "Step 9: Copying corrected RAW file to results folder..."
    cp "$WORKING_DIR/${OUTPUT_PREFIX}_corrected.raw" "$RESULTS_DIR/${OUTPUT_PREFIX}_corrected.raw"
    
    # ===== STEP 10: Generate carrier status =====
    echo "Step 10: Generating carrier status..."
    awk '
    BEGIN {
        OFS="\t"
    }
    NR==1 {
        print "IID", "Carrier_Status", "Variant_Count";
        next
    }
    {
        is_carrier = "Non-carrier"
        variant_count = 0
        for (i=7; i<=NF; i++) {
            if ($i > 0) {
                variant_count += $i
            }
        }
        if (variant_count > 0) {
            is_carrier = "Carrier"
        }
        print $2, is_carrier, variant_count;
    }' "$RESULTS_DIR/${OUTPUT_PREFIX}_corrected.raw" > "$RESULTS_DIR/${OUTPUT_PREFIX}_carrier_status_by_id.txt"
    
    # ===== Summary for this gene =====
    carrier_count=$(awk 'NR>1 && $2=="Carrier"' "$RESULTS_DIR/${OUTPUT_PREFIX}_carrier_status_by_id.txt" | wc -l)
    
    echo ""
    echo "✓ COMPLETE: $GENE_NAME"
    echo "  Chromosome: $GENE_CHROM"
    echo "  Pathogenic variants: $PATHOGENIC_COUNT"
    echo "  Carriers identified: $carrier_count"
    echo "  Output files:"
    echo "    - $RESULTS_DIR/${OUTPUT_PREFIX}_corrected.raw"
    echo "    - $RESULTS_DIR/${OUTPUT_PREFIX}_carrier_status_by_id.txt"
    
    # Append to summary CSV
    echo "$GENE_NAME,$GENE_CHROM,$PATHOGENIC_COUNT,$carrier_count" >> "$SUMMARY_CSV"
    
done

echo ""
echo "===== ALL GENES PROCESSED ====="
echo ""
echo "Working files (intermediate): $WORKING_DIR/"
echo "Final results: $RESULTS_DIR/"
echo "Summary CSV: $SUMMARY_CSV"
echo ""
echo "Summary CSV content:"
cat "$SUMMARY_CSV"